# HW2 - Self-Supervised Learning (Organized)

This notebook demonstrates the organized project structure for self-supervised learning experiments.
The code has been modularized into separate components for better maintainability.

## Project Structure

```
hw2/
├── src/
│   ├── models/
│   │   └── jigsaw_model.py      # JigsawNet model class
│   ├── datasets/
│   │   └── datasets.py          # Dataset classes (Rotation, Jigsaw, Test)
│   └── utils.py                 # Utility functions (permutation generation)
├── configs/
│   └── config.py                # Configuration and hyperparameters
├── scripts/
│   ├── train_rotation.py        # Rotation pretext task training
│   ├── train_jigsaw.py          # Jigsaw pretext task training
│   ├── finetune.py              # Fine-tuning script
│   └── inference.py             # Test set prediction generation
├── checkpoints/                 # Model weights (.pth files)
├── submissions/                 # Submission CSV files
├── notebooks/
│   └── hw2_original.ipynb       # Original notebook backup
└── data/                        # Dataset directory
```

## Setup and Imports

In [ ]:
import sys
import os

# Add the project root to Python path
sys.path.append(os.path.dirname(os.getcwd()))

import torch
import torch.nn as nn
from configs.config import Config

# Check device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project configuration loaded from: {Config.__module__}")

## Configuration Overview

In [ ]:
print("=== Project Configuration ===")
print(f"Data paths:")
print(f"  - Unlabeled: {Config.UNLABELED_PATH}")
print(f"  - Labeled: {Config.LABELED_PATH}")
print(f"  - Test: {Config.TEST_PATH}")
print(f"\nOutput paths:")
print(f"  - Checkpoints: {Config.CHECKPOINTS_DIR}")
print(f"  - Submissions: {Config.SUBMISSIONS_DIR}")
print(f"\nHyperparameters:")
print(f"  - Pretrain epochs: {Config.PRETRAIN_EPOCHS}")
print(f"  - Finetune epochs: {Config.FINETUNE_EPOCHS}")
print(f"  - Batch size: {Config.BATCH_SIZE}")
print(f"  - Learning rates: {Config.LEARNING_RATE_PRETRAIN} (pretrain), {Config.LEARNING_RATE_FINETUNE} (finetune)")
print(f"\nJigsaw parameters:")
print(f"  - Grid size: {Config.GRID_SIZE}x{Config.GRID_SIZE}")
print(f"  - Patch size: {Config.PATCH_SIZE}")
print(f"  - Number of permutations: {Config.N_PERMUTATIONS}")

## Running Training Scripts

Instead of running training code in the notebook, you can now use the organized scripts:

### 1. Train Rotation Pretext Task
```bash
cd scripts/
python train_rotation.py
```

### 2. Train Jigsaw Pretext Task
```bash
cd scripts/
python train_jigsaw.py
```

### 3. Fine-tune with Different Backbones
```bash
cd scripts/

# Fine-tune with rotation backbone
python finetune.py --method rotation

# Fine-tune with jigsaw backbone
python finetune.py --method jigsaw

# Train from scratch (no pretext task)
python finetune.py --method scratch
```

### 4. Generate Test Predictions
```bash
cd scripts/

# Generate predictions with rotation model
python inference.py --method rotation

# Generate predictions with jigsaw model
python inference.py --method jigsaw

# Generate predictions with scratch model
python inference.py --method scratch
```

## Quick Testing of Modular Components

You can still test individual components in the notebook:

In [ ]:
# Test dataset loading
from src.datasets.datasets import RotationDataset, JigsawDataset
from src.utils import generate_permutations
from torchvision import transforms

# Test transforms
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Testing modular components...")

# Test rotation dataset (if data exists)
if os.path.exists(Config.UNLABELED_PATH):
    rotation_dataset = RotationDataset(Config.UNLABELED_PATH, transform=test_transforms)
    print(f"Rotation dataset size: {len(rotation_dataset)}")
else:
    print(f"Unlabeled data not found at {Config.UNLABELED_PATH}")

# Test permutation generation
permutations = generate_permutations(10, Config.GRID_SIZE, "test_permutations_10.npy")
print(f"Generated {len(permutations)} permutations")
print(f"First permutation: {permutations[0]}")

In [ ]:
# Test model initialization
from src.models.jigsaw_model import JigsawNet
from torchvision import models

print("Testing model initialization...")

# Test JigsawNet
jigsaw_model = JigsawNet(num_permutations=Config.N_PERMUTATIONS, grid_size=Config.GRID_SIZE)
print(f"JigsawNet created with {sum(p.numel() for p in jigsaw_model.parameters())} parameters")

# Test standard ResNet
resnet_model = models.resnet18(num_classes=Config.NUM_CLASSES)
print(f"ResNet18 created with {sum(p.numel() for p in resnet_model.parameters())} parameters")

## Training Implementation

Now let's implement the full training pipeline in the notebook:

### 1. Rotation Pretext Task Training

In [ ]:
# Import additional necessary libraries for training
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, datasets
from tqdm.notebook import tqdm
import copy
import pandas as pd

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define transformations for the rotation pretext task
pretext_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Setup complete. Ready for rotation pretext training.")

In [ ]:
# Train Rotation Pretext Task
def train_rotation_pretext():
    print("Starting rotation pretext task training...")
    
    # Create Dataset and DataLoader
    pretrain_dataset = RotationDataset(Config.UNLABELED_PATH, transform=pretext_transforms)
    pretrain_loader = DataLoader(pretrain_dataset, batch_size=Config.BATCH_SIZE, 
                                shuffle=True, num_workers=2)
    
    print(f"Found {len(pretrain_dataset)} images for pre-training.")
    
    # Load ResNet-18 and modify the classifier for the pretext task
    pretext_model = models.resnet18(weights=None)  # Start from scratch
    num_features = pretext_model.fc.in_features
    pretext_model.fc = nn.Linear(num_features, 4)  # 4 classes for 4 rotation angles
    pretext_model = pretext_model.to(device)
    
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(pretext_model.parameters(), lr=Config.LEARNING_RATE_PRETRAIN)
    
    print("Starting self-supervised pre-training (Rotation)...")
    
    # Training loop
    for epoch in range(Config.PRETRAIN_EPOCHS):
        pretext_model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_predictions = 0
        
        progress_bar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{Config.PRETRAIN_EPOCHS}")
        
        for batch_idx, (images, labels) in enumerate(progress_bar):
            images, labels = images.to(device), labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = pretext_model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Update statistics
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
            
            # Update progress bar
            progress_bar.set_postfix(loss=loss.item(), acc=f"{(correct_predictions/total_predictions):.3f}")
        
        epoch_loss = running_loss / len(pretrain_loader.dataset)
        epoch_acc = correct_predictions / total_predictions
        print(f"Epoch {epoch+1}/{Config.PRETRAIN_EPOCHS} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
    
    print("Finished rotation pre-training.")
    
    # Save the backbone (feature extractor) weights
    # Remove the final fully connected layer ('fc') before saving
    backbone_weights = copy.deepcopy(pretext_model.state_dict())
    keys_to_remove = ["fc.weight", "fc.bias"]
    for key in keys_to_remove:
        if key in backbone_weights:
            del backbone_weights[key]
    
    torch.save(backbone_weights, Config.ROTATION_BACKBONE_PATH)
    print(f"Saved pre-trained backbone weights to '{Config.ROTATION_BACKBONE_PATH}'")
    
    return pretext_model

# Run rotation training (uncomment to execute)
# rotation_model = train_rotation_pretext()

### 2. Jigsaw Puzzle Pretext Task Training

In [ ]:
# Train Jigsaw Puzzle Pretext Task
def train_jigsaw_pretext():
    print("Starting jigsaw puzzle pretext task training...")
    
    # Generate or load permutations
    permutations = generate_permutations(Config.N_PERMUTATIONS, Config.GRID_SIZE, 
                                       Config.PERMUTATIONS_PATH)
    
    # Transformations for each image patch
    patch_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create dataset and dataloader
    pretrain_dataset = JigsawDataset(Config.UNLABELED_PATH, permutations, 
                                   grid_size=Config.GRID_SIZE, patch_size=Config.PATCH_SIZE, 
                                   transform=patch_transforms)
    pretrain_loader = DataLoader(pretrain_dataset, batch_size=Config.BATCH_SIZE, 
                                shuffle=True, num_workers=2)
    
    print(f"Found {len(pretrain_dataset)} images for pre-training.")
    
    # Initialize JigsawNet model
    jigsaw_model = JigsawNet(num_permutations=Config.N_PERMUTATIONS, 
                           grid_size=Config.GRID_SIZE).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(jigsaw_model.parameters(), lr=Config.LEARNING_RATE_PRETRAIN)
    
    print("Starting self-supervised pre-training (Jigsaw)...")
    
    # Training loop
    for epoch in range(Config.PRETRAIN_EPOCHS):
        jigsaw_model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_predictions = 0
        
        progress_bar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{Config.PRETRAIN_EPOCHS}")
        
        for batch_idx, (patches, labels) in enumerate(progress_bar):
            patches, labels = patches.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = jigsaw_model(patches)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * patches.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
            
            progress_bar.set_postfix(loss=loss.item(), acc=f"{(correct_predictions/total_predictions):.3f}")
        
        epoch_loss = running_loss / len(pretrain_loader.dataset)
        epoch_acc = correct_predictions / total_predictions
        print(f"Epoch {epoch+1}/{Config.PRETRAIN_EPOCHS} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")
    
    print("Pre-training completed.")
    
    # Save trained backbone weights
    torch.save(jigsaw_model.backbone.state_dict(), Config.JIGSAW_BACKBONE_PATH)
    print(f"Backbone weights saved to '{Config.JIGSAW_BACKBONE_PATH}'")
    
    return jigsaw_model

# Run jigsaw training (uncomment to execute)
# jigsaw_model = train_jigsaw_pretext()

### 3. Fine-tuning for Classification Task

In [ ]:
# Fine-tuning function for classification
def finetune_model(backbone_path, output_model_path, method_name="unknown"):
    """
    Fine-tune a model using a pre-trained backbone.
    
    Args:
        backbone_path (str): Path to the pre-trained backbone weights
        output_model_path (str): Path to save the best fine-tuned model
        method_name (str): Name of the pretext method (for logging)
    """
    print(f"Starting fine-tuning for {method_name} method...")
    
    # Transformations for the downstream classification task
    finetune_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create dataset using ImageFolder
    full_labeled_dataset = datasets.ImageFolder(Config.LABELED_PATH, transform=finetune_transforms)
    
    # Split labeled data into train and validation sets
    train_size = int(Config.TRAIN_VAL_SPLIT * len(full_labeled_dataset))
    val_size = len(full_labeled_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_labeled_dataset, [train_size, val_size])
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Get class names for later use
    class_names = full_labeled_dataset.classes
    print(f"Found {len(full_labeled_dataset)} labeled images in {len(class_names)} classes.")
    print("Classes:", class_names)
    
    # Initialize the final classification model
    finetune_model = models.resnet18(num_classes=Config.NUM_CLASSES)
    
    # Load the pre-trained backbone weights if available
    if backbone_path and os.path.exists(backbone_path):
        print(f"Loading pre-trained backbone weights from {backbone_path}...")
        try:
            finetune_model.load_state_dict(torch.load(backbone_path), strict=False)
            print("Successfully loaded backbone weights.")
        except Exception as e:
            print(f"Warning: Could not load backbone weights: {e}")
            print("Training from scratch...")
    else:
        print(f"Backbone weights not found at {backbone_path}. Training from scratch...")
    
    finetune_model = finetune_model.to(device)
    
    # Define loss and optimizer for fine-tuning
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(finetune_model.parameters(), lr=Config.LEARNING_RATE_FINETUNE)
    
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(finetune_model.state_dict())
    
    print(f"Starting supervised fine-tuning ({method_name})...")
    
    # Training loop
    for epoch in range(Config.FINETUNE_EPOCHS):
        # Training phase
        finetune_model.train()
        running_loss = 0.0
        correct_predictions = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.FINETUNE_EPOCHS} (Train)")
        
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct_predictions += (predicted == labels).sum().item()
            progress_bar.set_postfix(loss=loss.item())
        
        train_acc = correct_predictions / len(train_dataset)
        
        # Validation phase
        finetune_model.eval()
        val_correct = 0
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = finetune_model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = val_correct / len(val_dataset)
        print(f"Epoch {epoch+1}/{Config.FINETUNE_EPOCHS} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        
        # Save the best model based on validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(finetune_model.state_dict())
            torch.save(best_model_wts, output_model_path)
            print(f"New best model saved with validation accuracy: {best_val_acc:.4f}")
    
    print(f"Finished fine-tuning ({method_name}). Best Validation Accuracy: {best_val_acc:.4f}")
    return class_names, best_val_acc

print("Fine-tuning function defined.")

In [ ]:
# Fine-tune all methods
def run_all_experiments():
    results = {}
    
    print("="*60)
    print("RUNNING ALL EXPERIMENTS")
    print("="*60)
    
    # 1. Fine-tune with rotation backbone
    print("\n1. Fine-tuning with Rotation Backbone:")
    class_names, rotation_acc = finetune_model(
        Config.ROTATION_BACKBONE_PATH, 
        Config.BEST_MODEL_ROTATION_PATH, 
        "rotation"
    )
    results['rotation'] = rotation_acc
    
    # 2. Fine-tune with jigsaw backbone
    print("\n2. Fine-tuning with Jigsaw Backbone:")
    _, jigsaw_acc = finetune_model(
        Config.JIGSAW_BACKBONE_PATH, 
        Config.BEST_MODEL_JIGSAW_PATH, 
        "jigsaw"
    )
    results['jigsaw'] = jigsaw_acc
    
    # 3. Train from scratch (baseline)
    print("\n3. Training from Scratch (Baseline):")
    _, scratch_acc = finetune_model(
        None,  # No backbone weights
        Config.BEST_MODEL_PATH, 
        "scratch"
    )
    results['scratch'] = scratch_acc
    
    # Print results summary
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    for method, acc in results.items():
        print(f"{method.capitalize():12}: {acc:.4f} ({acc*100:.2f}%)")
    
    return results, class_names

# Uncomment to run all experiments
# results, class_names = run_all_experiments()

### 4. Inference and Submission Generation

In [ ]:
# Inference function
def generate_predictions(model_path, output_csv_path, class_names, method_name="unknown"):
    """
    Generate predictions on test set using a trained model.
    
    Args:
        model_path (str): Path to the trained model weights
        output_csv_path (str): Path to save the submission CSV
        class_names (list): List of class names
        method_name (str): Name of the method (for logging)
    """
    print(f"Generating predictions using {method_name} model...")
    
    # Use validation transforms for the test set (no random augmentations)
    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Create test dataset and dataloader
    test_dataset = TestDataset(Config.TEST_PATH, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
    
    print(f"Found {len(test_dataset)} test images")
    
    # Load the best model for inference
    final_model = models.resnet18(num_classes=Config.NUM_CLASSES)
    
    if os.path.exists(model_path):
        final_model.load_state_dict(torch.load(model_path))
        print(f"Loaded model from {model_path}")
    else:
        print(f"Warning: Model not found at {model_path}")
        return None
    
    final_model = final_model.to(device)
    final_model.eval()
    
    predictions = []
    image_ids = []
    
    print(f"Generating predictions on the test set...")
    with torch.no_grad():
        for images, fnames in tqdm(test_loader, desc="Inference"):
            images = images.to(device)
            outputs = final_model(images)
            _, predicted_indices = torch.max(outputs, 1)
            
            predictions.extend([class_names[i] for i in predicted_indices.cpu().numpy()])\n            image_ids.extend(fnames)
    
    # Create submission DataFrame
    submission_df = pd.DataFrame({
        'id': image_ids,
        'class': predictions
    })
    
    # Save to CSV
    submission_df.to_csv(output_csv_path, index=False)
    
    print(f"Submission file '{output_csv_path}' created successfully!")
    print(f"Generated {len(submission_df)} predictions")
    print(submission_df.head())
    
    return submission_df

print("Inference function defined.")

In [ ]:
# Generate all predictions
def generate_all_predictions(class_names):
    """Generate predictions for all trained models."""
    print("Generating predictions for all models...")
    
    # Predictions for rotation model
    rotation_df = generate_predictions(
        Config.BEST_MODEL_ROTATION_PATH,
        Config.SUBMISSION_ROTATION_PATH,
        class_names,
        "rotation"
    )
    
    # Predictions for jigsaw model
    jigsaw_df = generate_predictions(
        Config.BEST_MODEL_JIGSAW_PATH,
        Config.SUBMISSION_JIGSAW_PATH,
        class_names,
        "jigsaw"
    )
    
    # Predictions for scratch model
    scratch_df = generate_predictions(
        Config.BEST_MODEL_PATH,
        Config.SUBMISSION_PATH,
        class_names,
        "scratch"
    )
    
    return rotation_df, jigsaw_df, scratch_df

# Uncomment to generate all predictions
# rotation_df, jigsaw_df, scratch_df = generate_all_predictions(class_names)